## Import library

In [1]:
import pandas as pd
import numpy as np
import os
import json
import time
import joblib
from pathlib import Path
from typing import List, Dict, Any, Optional
from tqdm.auto import tqdm

# PDF Readers
from pypdf import PdfReader

# Langhchain framework
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# HF + Groq clients
from huggingface_hub import InferenceClient
from groq import Groq


import warnings
warnings.filterwarnings("ignore")

## Load API keys for credential key

In [2]:
from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(f"HF token loaded:  {HUGGINGFACE_API_KEY[:3]}...")
print(f"GROQ token loaded: {GROQ_API_KEY[:3]}...")

HF token loaded:  hf_...
GROQ token loaded: gsk...


## Load database

In [3]:
# Load dataset
df = pd.read_parquet("../credit_risk_production/database/data/merged_credit_risk_data.parquet")

# Load feature importance data
fi = pd.read_parquet("../credit_risk_production/database/data/features_data.parquet")

pd.set_option('display.max_columns', None)
print(f"Dataset & Feature Importance shape: {df.shape} & {fi.shape}")
df.head(3)

Dataset & Feature Importance shape: (51336, 87) & (51336, 47)


,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,Total_TL_opened_L12M,Tot_TL_closed_L12M,pct_tl_open_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,Auto_TL,CC_TL,Consumer_TL,Gold_TL,Home_TL,PL_TL,Secured_TL,Unsecured_TL,Other_TL,Age_Oldest_TL,Age_Newest_TL,time_since_recent_payment,time_since_first_deliquency,time_since_recent_deliquency,num_times_delinquent,max_delinquency_level,max_recent_level_of_deliq,num_deliq_6mts,num_deliq_12mts,num_deliq_6_12mts,max_deliq_6mts,max_deliq_12mts,num_times_30p_dpd,num_times_60p_dpd,num_std,num_std_6mts,num_std_12mts,num_sub,num_sub_6mts,num_sub_12mts,num_dbt,num_dbt_6mts,num_dbt_12mts,num_lss,num_lss_6mts,num_lss_12mts,recent_level_of_deliq,tot_enq,CC_enq,CC_enq_L6m,CC_enq_L12m,PL_enq,PL_enq_L6m,PL_enq_L12m,time_since_recent_enq,enq_L12m,enq_L6m,enq_L3m,MARITALSTATUS,EDUCATION,AGE,GENDER,NETMONTHLYINCOME,Time_With_Curr_Empr,pct_of_active_TLs_ever,pct_opened_TLs_L6m_of_L12m,pct_currentBal_all_TL,CC_utilization,CC_Flag,PL_utilization,PL_Flag,pct_PL_enq_L6m_of_L12m,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,max_unsec_exposure_inPct,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,5,4,1,0,0,0.000,0.0,0.2,0.8,0,0,0.00,0.0,0,0,0,0,1,0,4,1,4,0,72,18,549,35,15,11,29,29,0,0,0,-99999,-99999,0,0,21,5,11,0,0,0,0,0,0,0,0,0,29,6,0,0,0,6,0,0,566,0,0,0,Married,12TH,48,M,51000,114,0.2,0.0,0.798,-99999.0,0,0.798,1,0.0,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.0,0.0,1,0,1.00,0.0,0,0,0,1,0,0,0,0,1,0,7,7,47,-99999,-99999,0,-99999,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,209,1,0,0,Single,GRADUATE,23,F,19000,50,1.0,0.0,0.370,-99999.0,0,-99999.000,0,0.0,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.0,0.0,2,0,0.25,0.0,1,1,0,6,1,0,0,2,6,0,47,2,302,11,3,9,25,25,1,9,8,25,25,0,0,10,5,10,0,0,0,0,0,0,0,0,0,25,4,0,0,0,0,0,0,587,0,0,0,Married,SSC,40,M,18,191,1.0,0.5,0.585,-99999.0,0,-99999.000,0,0.0,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2


## Load ML Models

In [4]:
MODEL_DIR = Path("../credit_risk_production/models/credit_risk")
MODEL_SUBDIR = MODEL_DIR / "ml_credit_risk"

ml_models = {
    model_file.stem: joblib.load(model_file)
    for model_file in MODEL_SUBDIR.glob("*.joblib")
}
model_bundle = joblib.load(MODEL_DIR / "model_bundle.joblib")
parameters = json.loads((MODEL_DIR / "params_credit_risk" / "best_parameters.json").read_text())
metadata = json.loads((MODEL_DIR / "metadata_credit_risk" / "metadata.json").read_text())
metrics_df = pd.read_csv(MODEL_DIR / "metrics_credit_risk" / "model_metrics.csv")

print(f"Loaded ML models: {list(ml_models.keys())}")
print(f"Loaded best parameters: {parameters}")
print(f"Loaded metadata: {metadata}")
print(f"Loaded model bundle: {model_bundle}")
print(metrics_df)

Loaded ML models: ['k_nearest_neighbors', 'logistic_regression', 'gradient_boosting', 'random_forest', 'xgboost', 'decision_tree']
Loaded best parameters: {'Logistic Regression': {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'C': 100}, 'Random Forest': {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30}, 'Gradient Boosting': {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 3, 'learning_rate': 0.01}, 'XGBoost': {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}, 'K-Nearest Neighbors': {'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}, 'Decision Tree': {'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 10}}
Loaded metadata: {'model_files': {'Logistic Regression': 'logistic_regression.joblib', 'Random Forest': 'random_forest.joblib', 'Gradient Boosting': 'gradient_boosting.joblib', 'XGBoost': 'xgboost.joblib', 'K-Nearest Neighbors': 'k_

## Identify the actual trained estimator inside the bundle


In [5]:
def pick_estimator(bundle):
    if not isinstance(bundle, dict):
        return bundle

    for key in ['models', 'scaler', 'label_encoders', 'feature_columns', 'class_labels']:
        if key in bundle:
            return bundle[key]

    # Fallback: first object that has predict
    for v in bundle.values():
        if hasattr(v, 'predict'):
            return v

    raise ValueError("No estimator found in the bundle.")

risk_model = pick_estimator(model_bundle)
print("✅ Risk model:", type(risk_model).__name__)

✅ Risk model: dict


## Defin one best model or model chosen

In [23]:
risk_model = model_bundle['models']['Gradient Boosting']
scaler = model_bundle['scaler']
label_encoders = model_bundle['label_encoders']

print(f"Selected Model: {type(risk_model).__name__}")

Selected Model: GradientBoostingClassifier


## Identify feature columns used by the model

In [ ]:
if isinstance(model_bundle, dict) and "feature_columns" in model_bundle:
    MODEL_FEATURES = model_bundle["feature_columns"]
elif "feature_columns" in metadata:
    MODEL_FEATURES = metadata["feature_columns"]
else:
    raise ValueError("Feature names not found in model bundle or metadata.")

print(f"Number of model features: {len(MODEL_FEATURES)}")
print(f"Model features: {MODEL_FEATURES}")

Number of model features: 47
Model features: ['num_dbt_12mts', 'pct_tl_open_L12M', 'num_lss_12mts', 'num_lss_6mts', 'pct_tl_closed_L12M', 'pct_tl_closed_L6M', 'time_since_first_deliquency', 'MARITALSTATUS', 'time_since_recent_deliquency', 'first_prod_enq2', 'PL_Flag', 'tot_enq', 'GENDER', 'num_sub', 'last_prod_enq2', 'num_times_60p_dpd', 'Tot_TL_closed_L6M', 'pct_opened_TLs_L6m_of_L12m', 'GL_Flag', 'max_deliq_6mts', 'pct_currentBal_all_TL', 'num_sub_12mts', 'EDUCATION', 'max_recent_level_of_deliq', 'max_delinquency_level', 'pct_tl_open_L6M', 'num_lss', 'CC_Flag', 'num_times_delinquent', 'num_sub_6mts', 'max_unsec_exposure_inPct', 'HL_Flag', 'Tot_TL_closed_L12M', 'Credit_Score', 'PL_utilization', 'num_dbt_6mts', 'CC_utilization', 'num_std', 'Time_With_Curr_Empr', 'Total_TL_opened_L6M', 'time_since_recent_payment', 'Tot_Missed_Pmnt', 'NETMONTHLYINCOME', 'recent_level_of_deliq', 'num_dbt', 'AGE', 'max_deliq_12mts']


## Scale score for one customer

In [40]:
TARGET = "Approved_Flag"

def score_customer(row: pd.Series) -> Dict[str, Any]:
    """Return ML risk probability for a single customer row."""
    df_input = pd.DataFrame([row[MODEL_FEATURES].to_dict()])

    # Apply LableEncoders to categorical features
    for col, le in label_encoders.items():
        if col in df_input.columns:
            # Handle unseen categories labels
            val = str(df_input[col].iloc[0])
            if val in le.classes_:
                df_input[col] = le.transform([val])
            else:
                df_input[col] = 0 # Fallback index for unseen categories

    # Apply StandardScaler
    X_scaled = scaler.transform(df_input[MODEL_FEATURES])

    # Predict class probabilities and label
    proba = risk_model.predict_proba(X_scaled)[0]
    predicted_class = int(risk_model.predict(X_scaled)[0])

    # Map probability per class label [0, 1, 2, 3]
    class_probabilities = {
        f"class_{cls}_prob": round(float(prob), 4)
        for cls, prob in zip(risk_model.classes_, proba)
    }

    return {
        "model_used": type(risk_model).__name__,
        "predicted_flag": predicted_class,
        "class_probabilities": class_probabilities,
        "primary_risk_probability": round(float(proba[predicted_class]), 4)
    }

# Usage
sample = df.iloc[0]
print(f"Score on customer: {score_customer(sample)}")

Score on customer: {'model_used': 'GradientBoostingClassifier', 'predicted_flag': 1, 'class_probabilities': {'class_0_prob': 0.0283, 'class_1_prob': 0.9073, 'class_2_prob': 0.0358, 'class_3_prob': 0.0286}, 'primary_risk_probability': 0.9073}


# 📚 Document Ingestion — PDF → Chunks (per-document strategy)

## Read all PDFs

In [43]:
PDF_DIR = Path("../credit_risk_production/database/pdf")

pdf_files = {
    "delinquency":  PDF_DIR / "Delinquency_Classification .pdf",
    "fraud":        PDF_DIR / "Fraud_Typologies_and_Red Flags .pdf",
    "regulatory":   PDF_DIR / "Regulatory_Risk Policy Core .pdf",
    "scorecard":    PDF_DIR / "Scorecard_Cut-off Policy.pdf"
}

raw_texts: Dict[str, str] = {}

for name, path in pdf_files.items():
    reader = PdfReader(str(path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    raw_texts[name] = text
    print(f"{name:12s} | pages={len(reader.pages):3d} | chars={len(text):,}")

delinquency  | pages= 93 | chars=191,238
fraud        | pages=  9 | chars=25,113
regulatory   | pages= 11 | chars=34,419
scorecard    | pages=178 | chars=465,508


## Chunking strategy to retrieve Document augmentation for each PDFs to enrich vocab on LLM
- #### Strategy 1: Delinquency Classification — rule-based chunking
- #### Strategy 2: Fraud Typologies — semantic paragraph chunking
- #### Strategy 3: Regulatory Policy — recursive with heading preservation
- #### Strategy 4: Scorecard Cut-off — table-aware chunking
- #### Final Strategy: Combine all chunks

#### Strategy 1: Delinquency Classification — rule-based chunking

In [ ]:
def chunk_delinquency(text: str) -> List[Document]:
    """Split by classification headings, falls back to paragraph split."""
    keywords = ["Standard", "Sub-standard", "Substandrad", "Doubtful", "Loss"]

    # Crude split: find the index of each keyword and slice
    positions = []
    for kw in keywords:
        idx = text.lower().find(kw.lower())
        if idx != -1:
            positions.append((idx, kw))
    positions.sort()

    docs: List[Document] = []
    if not positions:
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)